# Batch 7 — final assembly and confirmation (validation only)

**Runtime:** **L4 GPU**. Runtime → Run all. Run only after batches 4, 5 and 6 have finished. About **2 h 30 min – 4 h**, depending
on what is kept (nothing kept = about 1 h; a kept E07 makes the walk-forward slower). If Colab disconnects, Run all again: finished
runs are skipped and every decision is re-derived from the saved files.

Every decision below is made by `scripts/select_final.py` and `scripts/posthoc_eval.py` with rules fixed before batches 4–6 ran
(`claude/batches-4-7-design-2026-10-16.md`):

1. Changes kept in batches 5–6 (counted gain, no counted loss vs E21); conflicts resolved by fixed rules.
2. 2+ kept → the stacked run is trained; the final configuration is the best of the stacked run (if it has no counted loss) and the kept single runs.
3. E13: older rows (before 2022) at loss weight 0.5, on the final configuration; kept by the same rule.
4. E25: 6 more seeds of the final configuration → 9-seed ensemble in 3 groups (reported beside the single-model score).
5. Walk-forward of the final configuration on the same 4 windows (reused from batch 3 if the final configuration is still E21).
6. Blend / calibration from batch 4 re-applied per horizon, chosen on the walk-forward windows only; pass bar and confirmation.

Outputs in `MyDrive/Senior/results/batch7/`. Test block stays sealed.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, json, os, subprocess, sys
for f in ['scripts/posthoc_eval.py', 'scripts/select_final.py', 'scripts/factors_tabular.py', 'src/alpha158.py', 'src/features_vol.py']:
    assert os.path.exists(f), f'stop: push the batch 4-7 code first ({f} missing)'
for f in ['scripts/baselines_deep.py', 'scripts/posthoc_eval.py', 'src/multiscale_patchtst.py']:
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
FOLDS = [('2022H2', '2022-07-01', '2022-12-31'), ('2023H1', '2023-01-01', '2023-06-30'),
         ('2023H2', '2023-07-01', '2023-12-31'), ('2024H1', '2024-01-01', '2024-06-30')]
B3, B4, B5, B6, B7 = (f'{OUT_DIR}/batch{i}' for i in (3, 4, 5, 6, 7))
E21_JSON = f'{B3}/e21_stacked.json'
need = [E21_JSON, E21_JSON.replace('.json', '_preds.csv.gz'), f'{OUT_DIR}/xgb_e03.json', f'{OUT_DIR}/xgb_e03_preds.csv.gz']
need += [f'{B3}/e22/{f}_deep{s}' for f, _, _ in FOLDS for s in ('.json', '_preds.csv.gz')]
need += [f'{OUT_DIR}/e05/{f}_xgb{s}' for f, _, _ in FOLDS for s in ('.json', '_preds.csv.gz')]
need += [f'{B4}/batch4_summary.json', f'{B5}/batch5_summary.json', f'{B6}/batch6_summary.json']
for f in need:
    assert os.path.exists(f), f'stop: missing {f}'
for d in [B7, f'{B7}/wf']:
    os.makedirs(d, exist_ok=True)
print('earlier files found; batch 7 outputs go to', B7)

In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
# Tests; the notebook stops here if any fails (about 5 minutes)
for t in ['tests/test_pipeline.py', 'tests/test_train_options.py', 'tests/test_select_final.py', 'tests/test_posthoc.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l)[-1500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed\n' + r.stderr[-2000:]

In [ ]:
E21 = ['--models', 'multiscale_sector', '--news', 'data/derived/sentiment_scores.csv', '--extra-features', 'e03',
       '--lr', '1e-4', '--dropout', '0.2', '--max-epochs', '30', '--patience', '10', '--cls-only', '--stop-rule', 'every',
       '--no-revin']
def done(path, n=3):
    try:
        return len(json.load(open(path))['models']['multiscale_sector']['runs']) == n
    except Exception:
        return False
def run(label, args, out, n=3):
    if done(out, n):
        print(f'=== {label}: already done'); return
    print(f'=== {label}'); sys.stdout.flush()
    subprocess.run(['python', 'scripts/baselines_deep.py', *args, '--out', out], check=True)
def opt(args, **kw):
    a = list(args)
    for k, v in kw.items():
        k = '--' + k.replace('_', '-')
        if k in a: a[a.index(k) + 1] = str(v)
        else: a += [k, str(v)]
    return a

In [ ]:
# 1. Which changes are kept, and how conflicts resolve
CFG = f'{B7}/final_config.json'
!python scripts/select_final.py select --summary "$B5/batch5_summary.json" "$B6/batch6_summary.json" --out "$CFG"
cfg = json.load(open(CFG))

In [ ]:
# 2. Stacked run (only if 2+ changes are kept)
if cfg['needs_stacked_run']:
    run('STACK ' + '+'.join(cfg['used']), cfg['stacked_args'], f'{B7}/stack.json')
    rows = {r['label']: r['file'] for p in (f'{B5}/batch5_summary.json', f'{B6}/batch6_summary.json')
            for r in json.load(open(p))['rows']}
    extra = ' '.join(f'--run {lb} "{rows[lb]}"' for lb in cfg['used'])
    !python scripts/compare_batch.py --per-horizon-best --control-label E21 --control "$E21_JSON" --run STACK "$B7/stack.json" {extra} --out "$B7/stack_summary.json"
    !python scripts/select_final.py stacked --summary "$B7/stack_summary.json" --config "$CFG"
cfg = json.load(open(CFG))
FINAL_JSON = cfg['final_run'] or E21_JSON
print('final configuration so far:', cfg['final_label'], '\nargs:', ' '.join(cfg['final_args']), '\nrun file:', FINAL_JSON)

In [ ]:
# 3. E13: older rows at weight 0.5, on the final configuration
run('E13 recent weighting', opt(cfg['final_args'], recent_weight=0.5, recent_from='2022-01-01'), f'{B7}/e13_recent.json')
!python scripts/compare_batch.py --per-horizon-best --control-label FINAL --control "$FINAL_JSON" --run E13 "$B7/e13_recent.json" --out "$B7/e13_summary.json"
!python scripts/select_final.py e13 --summary "$B7/e13_summary.json" --config "$CFG"
cfg = json.load(open(CFG))
FINAL_JSON = cfg['final_run'] or E21_JSON
FINAL_ARGS, FINAL_LABEL = cfg['final_args'], cfg['final_label']
print('FINAL:', FINAL_LABEL, '|', ' '.join(FINAL_ARGS))

In [ ]:
# 4. E25: 6 more seeds -> 9-seed ensemble, 3 groups (E02 method), per-horizon best epochs
run('E25 seeds 18-23', FINAL_ARGS + ['--seeds', *map(str, range(20261018, 20261024))], f'{B7}/e25_more_seeds.json', n=6)
P0, P1 = FINAL_JSON.replace('.json', '_preds.csv.gz'), f'{B7}/e25_more_seeds_preds.csv.gz'
!python scripts/seed_ensemble.py --preds "$P0" "$P1" --model multiscale_sector_ph --groups 20261015,20261016,20261017 20261018,20261019,20261020 20261021,20261022,20261023 --out "$B7/e25_ensemble.json"

In [ ]:
# 5. Walk-forward of the final configuration (reuses batch 3's E22 if the final configuration is still E21)
if FINAL_LABEL == 'E21':
    WF = {f: f'{B3}/e22/{f}_deep.json' for f, _, _ in FOLDS}
else:
    WF = {f: f'{B7}/wf/{f}_deep.json' for f, _, _ in FOLDS}
    for f, s, e in FOLDS:
        run(f'walk-forward {f}', FINAL_ARGS + ['--eval-start', s, '--eval-end', e], WF[f])
A = ' '.join(f'--fold {f} "{WF[f]}" "{OUT_DIR}/e05/{f}_xgb.json"' for f, _, _ in FOLDS) + f' --fold official "{FINAL_JSON}" "{OUT_DIR}/xgb_e03.json"'
!python scripts/summarize_walk_forward.py --experiment FINAL --per-horizon-best {A} --out "$B7/final_walkforward.json"

In [ ]:
# 6. Blend / calibration (batch 4 rules), pass bar, confirmation and 95% intervals
DW = ' '.join(f'--deep-window {f} "{WF[f].replace(".json", "_preds.csv.gz")}"' for f, _, _ in FOLDS)
XW = ' '.join(f'--xgb-window {f} "{OUT_DIR}/e05/{f}_xgb_preds.csv.gz"' for f, _, _ in FOLDS)
WJ = ' '.join(f'--window-json {f} "{WF[f]}"' for f, _, _ in FOLDS)
XJ = ' '.join(f'--xgb-json {f} "{OUT_DIR}/e05/{f}_xgb.json"' for f, _, _ in FOLDS)
P0 = FINAL_JSON.replace('.json', '_preds.csv.gz')
!python scripts/posthoc_eval.py final --label "$FINAL_LABEL" --deep-official "$P0" {DW} --xgb-official "$OUT_DIR/xgb_e03_preds.csv.gz" {XW} \
    --e29-dir "$B4/e29" {WJ} {XJ} --batch4 "$B4/batch4_summary.json" --out "$B7/final_summary.json"